In [115]:
import pandas as pd
import numpy as np

### Na końcu tego pliku zapisujemy zawsze aktualne wyczyszczone dane i z nich korzystamy w innych notebookach np first_data_analysis.ipynb

**clean_data.csv** to ramka w której znajduje się wszystkie informacje <br>
**data_coded.csv** już kodujemy niektóre cechy i bardziej format jaki dla modelu

Na razie tylko obserwacja braków danych, jakieś proste operacje na kolumnach.  
**TO DO:** Trzeba będzie każdą kolumnę, która ma wiele wartości po przecinku jakoś inaczej reprezentować (albo usunąć i pozostawić po jednej wartości np gatunku albo jakoś pomyśleć jak to zakodować).
Wizualizacje czasem da sie wykonywać bez tego nawet, ale potem musimy mieć to wyczyszczone i zakodowane na przyszłość do modelów więc trzeba to zrobić

In [116]:
data = pd.read_csv(('../data/merged/merged_data.csv'))

In [117]:
data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')
data['year'] = data['year'].astype(int)

In [118]:
data.isna().sum()

tmdbId                    0
title                     0
budget                    0
revenue                   0
release_date              0
runtime                   0
original_language         0
popularity                0
vote_average              0
vote_count                0
origin_countries        370
spoken_languages        341
year                      0
keywords               3179
genres                   89
director_id              82
director_name            82
director_popularity      82
director_gender          82
writer_id               923
writer_name             923
writer_popularity       923
writer_gender           923
actor4_id               645
actor4_name             645
actor4_popularity       645
actor4_gender           645
actor2_id               256
actor2_name             256
actor2_popularity       256
actor2_gender           256
actor1_id               129
actor1_name             129
actor1_popularity       129
actor1_gender           129
actor5_id           

In [119]:
# zamiana wartości 0 w budżecie na wartości brakujące
data['budget'] = data['budget'].replace(0, np.nan)
data['budget_adjusted'] = data['budget_adjusted'].replace(0, np.nan)

In [120]:
data['budget'].isna().sum()

np.int64(8290)

In [121]:
print(data['revenue_adjusted'].quantile(0.05))
print(data['budget_adjusted'].quantile(0.05))

9913.57
78721.5


In [122]:
# usuwamy filmy o bardzo małym przychodzie (prawdopodobnie błędne dane)
data = data[data['revenue_adjusted'] > 10000]

# nierealnie mały budżet zamieniamy na NaN
data['budget_adjusted'] = data['budget_adjusted'].apply(lambda x: np.nan if x < 10000 else x)
data['budget'] = data['budget'].apply(lambda x: np.nan if x < 10000 else x)
data.head()

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor5_id,actor5_name,actor5_popularity,actor5_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender,budget_adjusted,revenue_adjusted
0,5.0,Four Rooms,4000000.0,4257354,1995-12-09,98,en,3.8441,5.892,2810,...,3125.0,Madonna,2.6797,1.0,3130.0,Jennifer Beals,3.9675,1.0,8.449948e+06,8.993604e+06
1,6.0,Judgment Night,21000000.0,12136938,1993-10-15,109,en,2.2315,6.469,368,...,9777.0,Cuba Gooding Jr.,2.5604,2.0,12799.0,Jeremy Piven,2.7546,2.0,4.678756e+07,2.704085e+07
2,11.0,Star Wars,11000000.0,775398007,1977-05-25,121,en,20.6912,8.200,22061,...,12248.0,Alec Guinness,2.0063,2.0,4.0,Carrie Fisher,2.5504,1.0,5.843850e+07,4.119372e+09
3,12.0,Finding Nemo,94000000.0,940335536,2003-05-30,100,en,16.5979,7.817,20364,...,12.0,Alexander Gould,2.0017,2.0,118.0,Geoffrey Rush,2.4578,2.0,1.644709e+08,1.645296e+09
4,13.0,Forrest Gump,55000000.0,677387716,1994-06-23,142,en,26.0309,8.500,29387,...,32.0,Robin Wright,3.9461,1.0,35.0,Sally Field,6.1880,1.0,1.194795e+08,1.471527e+09


In [123]:
# usunięcie filmów o czasie krótszym niż 10min i jednego filmu dłuższego niż 600min
data = data[(data['runtime'] > 10) & (data['runtime'] < 600)]

In [124]:
data.shape

(19237, 45)

In [125]:
data.isna().sum()

tmdbId                    0
title                     0
budget                 7639
revenue                   0
release_date              0
runtime                   0
original_language         0
popularity                0
vote_average              0
vote_count                0
origin_countries        211
spoken_languages        232
year                      0
keywords               2678
genres                   41
director_id              30
director_name            30
director_popularity      30
director_gender          30
writer_id               687
writer_name             687
writer_popularity       687
writer_gender           687
actor4_id               403
actor4_name             403
actor4_popularity       403
actor4_gender           403
actor2_id               137
actor2_name             137
actor2_popularity       137
actor2_gender           137
actor1_id                70
actor1_name              70
actor1_popularity        70
actor1_gender            70
actor5_id           

In [126]:
# tyle pozostałoby filmów gdyby wyrzucić wszystkie brakujące dane
data.dropna().shape[0]

10044

In [127]:
# dodanie kolumny odpowiadającej za kwartał 
data['quarter'] = data['release_date'].dt.quarter

In [128]:
# dodanie kolumny epoka dzielącej czas na okresy 20-letnie (1950-69, 1970-89, 1990-09, 2010-obecnie)

bins = [1950, 1970, 1990, 2010, 2030] 
labels = ['1950-1969', '1970-1989', '1990-2009', '2010-obecnie']

data['epoka'] = pd.cut(data['year'], bins=bins, labels=labels, right=False)
data['epoka'] = pd.Categorical(data['epoka'], categories=labels, ordered=True)

### Czyszczenie info o aktorach i ludziach

In [129]:
kolumny_aktorzy = [
    'actor1_popularity', 'actor2_popularity', 'actor3_popularity', 
    'actor4_popularity', 'actor5_popularity'
]

data['total_actors_popularity'] = data[kolumny_aktorzy].fillna(0).sum(axis=1)

kolumny_tworcy = ['director_popularity', 'writer_popularity']
data['total_crew_popularity'] = data[kolumny_tworcy].fillna(0).sum(axis=1)

In [130]:
data[['title', 'actor1_name','total_actors_popularity']].sort_values('total_actors_popularity', ascending=False)

,title,actor1_name,total_actors_popularity
9667,The Expendables 2,Chuck Norris,159.7070
3215,Operation Condor,Jackie Chan,109.9815
10601,Armour of God 3: Chinese Zodiac,Jackie Chan,109.9538
13692,The Fate of the Furious,Jason Statham,99.5044
919,Vanilla Sky,Tom Cruise,98.4453
...,...,...,...
5288,Zoo,NaN,0.0000
10090,Detropia,NaN,0.0000
17948,My Journey Through French Cinema,NaN,0.0000
10092,Escape Fire: The Fight to Rescue American Heal...,NaN,0.0000


In [131]:
data[['title','director_name', 'writer_name', 'total_crew_popularity']].sort_values('total_crew_popularity', ascending=False)

,title,director_name,writer_name,total_crew_popularity
20110,Dhurandhar,Aditya Dhar,Aditya Dhar,53.4862
11610,Lost River,Ryan Gosling,Ryan Gosling,49.9696
8931,Larry Crowne,Tom Hanks,Tom Hanks,30.7904
2382,That Thing You Do!,Tom Hanks,Tom Hanks,30.7904
19918,Article 370,Aditya Dhar,Arjun Dhawan,26.6230
...,...,...,...,...
13533,The Adventures of Marshall Art,NaN,NaN,0.0000
16509,Rang Panjab,NaN,NaN,0.0000
6574,Reform School Girls,Tom DeSimone,Tom DeSimone,0.0000
18929,AEW All Out 2022,NaN,NaN,0.0000


### Czyszczenie gatunków

In [132]:
# czyszczenie gatunków
def usun_falszywy_gatunek(tekst):
    
    if pd.isna(tekst):
        return tekst
    
    lista_gatunkow = tekst.split(',')

    czyste_gatunki = [g.strip() for g in lista_gatunkow if g.strip() != '(no genres listed)']
    
    if len(czyste_gatunki) == 0:
        return np.nan
    else:
        return ', '.join(czyste_gatunki)

In [133]:
data['genres'] = data['genres'].apply(usun_falszywy_gatunek)
data['genres'] = data['genres'].str.replace('science fiction', 'sci-fi', regex=False)
data['genres'] = data['genres'].str.replace('sciencefiction', 'sci-fi', regex=False)

data['genres'] = data['genres'].str.replace('tv movie', 'tvmovie', regex=False)

data['genres'] = data['genres'].str.replace('music', 'musical', regex=False)
data['genres'] = data['genres'].str.replace('musicalal', 'musical', regex=False)

In [134]:
# dodanie kolumny main_genre

# bo przyda się to do wykresów i staystyk żeby czasem brać po jednym gatunku na film
# jeśli pierwszy gatunek ogólny 'drama' to bierzemy drugi

def wybierz_lepszy_gatunek(genre_string):

    if pd.isna(genre_string):
        return np.nan
    
    gatunki = genre_string.split(',')
    pierwszy_gatunek = gatunki[0].strip()
    
    if pierwszy_gatunek == 'drama' and len(gatunki) > 1:
  
        return gatunki[1].strip()
    else:

        return pierwszy_gatunek

In [135]:
data = data.copy()
data['main_genre'] = data['genres'].apply(wybierz_lepszy_gatunek)

# usunięcie tvmovie bo nie pasują do filmów kinowych
# oraz children bo to też dla tmdb filmy na DVD dla dzieci

data = data[~data['main_genre'].isin(['tvmovie', 'children'])]
data['main_genre'].value_counts()

main_genre
comedy         4733
action         2502
romance        1549
drama          1496
horror         1387
thriller       1146
crime           940
animation       915
adventure       840
documentary     633
family          568
history         525
fantasy         426
sci-fi          423
musical         357
mystery         317
war             277
western         150
Name: count, dtype: int64

In [136]:
# Treaz zliczam wszystkie gatunki występujące (nie tylko main_genre)
wszystkie_gatunki = data['genres'].str.split(',').explode()
wszystkie_gatunki = wszystkie_gatunki.str.strip()

niechciane = ['(no genres listed)', '', None]
wszystkie_gatunki = wszystkie_gatunki[~wszystkie_gatunki.isin(niechciane)]

liczba_unikalnych = wszystkie_gatunki.nunique()
statystyki_gatunkowe = wszystkie_gatunki.value_counts()

statystyki_gatunkowe
# jest ich 22

genres
drama          9885
comedy         7208
thriller       4497
action         4174
romance        3777
adventure      2893
crime          2859
sci-fi         2424
horror         2190
fantasy        1913
family         1875
mystery        1587
animation      1395
history        1020
musical         925
documentary     746
children        740
war             691
western         282
imax            158
tvmovie          34
film-noir        23
Name: count, dtype: int64

In [137]:
# kodowanie gatunków 0-1

genres_encoded = data['genres'].str.get_dummies(sep=', ')
data_coded = pd.concat([data, genres_encoded], axis=1)

In [138]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# lista_gatunkow = list(genres_encoded.columns)
# korelacje = data_coded[lista_gatunkow].corrwith(data_coded['revenue_adjusted']).sort_values(ascending=False)

# plt.figure(figsize=(12, 10))

# sns.barplot(x=korelacje.values, y=korelacje.index, palette='coolwarm')


# plt.title('Korelacja gatunku i przychodu', fontsize = 22, fontweight='bold')
# plt.xlabel('Współczynnik korelacji', fontsize = 16)
# plt.ylabel('Gatunek', fontsize = 1)

# plt.axvline(x=0, color='black', linestyle='-', linewidth=1)

# plt.xticks(fontsize=16)
# plt.yticks(fontsize=16)

# plt.show()

In [139]:
kolumny_aktorzy = [
    'actor1_popularity', 'actor2_popularity', 'actor3_popularity', 
    'actor4_popularity', 'actor5_popularity'
]

data['total_actors_popularity'] = data[kolumny_aktorzy].fillna(0).sum(axis=1)

kolumny_tworcy = ['director_popularity', 'writer_popularity']
data['total_crew_popularity'] = data[kolumny_tworcy].fillna(0).sum(axis=1)

### Zapis ramki

In [140]:
data.to_csv("../data/merged/clean_data.csv", index = False)
data_coded.to_csv("../data/merged/data_coded.csv", index = False)